In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [2]:
df = pd.read_csv('/content/Tweets.csv')

df = df[['text', 'airline_sentiment']]

df.head()

,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [3]:
df.isnull().sum()

,0
text,0
airline_sentiment,0


In [4]:
df = df.dropna()

In [5]:
df.duplicated().sum()

np.int64(188)

In [6]:
df = df.drop_duplicates()

In [7]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'http\S+', '', text)

    text = re.sub(r'@\w+', '', text)

    text = re.sub(r'[^a-z\s]', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [8]:
df['clean_text'] = df['text'].apply(clean_text)

df[['text', 'clean_text']].head()

,text,clean_text
0,@VirginAmerica What @dhepburn said.,what said
1,@VirginAmerica plus you've added commercials t...,plus youve added commercials to the experience...
2,@VirginAmerica I didn't today... Must mean I n...,i didnt today must mean i need to take another...
3,@VirginAmerica it's really aggressive to blast...,its really aggressive to blast obnoxious enter...
4,@VirginAmerica and it's a really big bad thing...,and its a really big bad thing about it


In [9]:
df['airline_sentiment'].value_counts()

,count
airline_sentiment,
negative,9087
neutral,3067
positive,2298


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['airline_sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=df['airline_sentiment']
)

In [11]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train)

X_test_tfidf = tfidf.transform(X_test)

In [12]:
X_train_tfidf.shape

(11561, 5000)

In [13]:
pd.DataFrame(
    X_train_tfidf[:5].toarray(),
    columns=tfidf.get_feature_names_out()
)

,aa,aaaand,aaadvantage,aaalwayslate,aaba,aacom,aadavantage,aadelay,aadfw,aadv,...,zero,zip,zippers,zone,zones,zoom,zrh,zukes,zurich,zurichnew
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
model = LogisticRegression(
    max_iter=1000
)

model.fit(
    X_train_tfidf,
    y_train
)

LogisticRegression(max_iter=1000)

In [15]:
y_pred = model.predict(
    X_test_tfidf
)

In [16]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred,
    average='weighted'
)

In [17]:
pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Score'],
    'Score': [accuracy, f1]
})

,Metric,Score
0,Accuracy,0.769630
1,F1 Score,0.752956


In [18]:
feature_names = np.array(
    tfidf.get_feature_names_out()
)

top_words = []

for i, sentiment in enumerate(model.classes_):

    indexes = np.argsort(
        model.coef_[i]
    )[-10:][::-1]

    for index in indexes:

        top_words.append({
            'Class': sentiment,
            'Word': feature_names[index],
            'Coefficient': model.coef_[i][index]
        })

In [19]:
top_words_df = pd.DataFrame(
    top_words
)

top_words_df

,Class,Word,Coefficient
0,negative,hours,3.330297
1,negative,worst,3.160954
2,negative,hour,2.801654
3,negative,hrs,2.605205
4,negative,delayed,2.602748
5,negative,luggage,2.601936
6,negative,hold,2.544484
7,negative,cancelled,2.512910
8,negative,fail,2.149981
9,negative,fix,2.137586
